# Task 4: Cost-Optimized LLM Batch Analysis: Bridge Project Greenville

### Solves: Token Limit (400), Rate Limit (429), AND High API Cost

**Key Design Principle:**
The LLM gets value from TEXT grounding (labels, dimensions, annotations), not from raw line/geometry coordinates.
We aggressively prune low-value data while preserving 100% of text grounding.


In [ ]:
# @title 1. Install Gemini SDK and Setup
# Installs the official, consolidated standard package
!pip install -q -U google-genai

from google import genai
from google.colab import userdata
import os
import json
import time
import re
import zipfile
from PIL import Image

# Setup API Key using your Colab Userdata Secrets
API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=API_KEY)

# Force retention of your designated Gemini 3 Flash Preview model
MODEL_NAME = 'gemini-3-flash-preview'

# =================================================================
# ⚙️ TRUE STANDARD PRODUCTION TARIFFS (Gemini 3 Flash Preview)
# =================================================================
TRUE_INPUT_PRICE_PER_1M = 0.50   # $0.50 per 1,000,000 tokens
TRUE_OUTPUT_PRICE_PER_1M = 3.00  # $3.00 per 1,000,000 tokens

print(f"✅ New GenAI SDK Configured. Model locked to: {MODEL_NAME}")

In [ ]:
# @title 2. Upload and Unzip Task 1 Data
from google.colab import files

print("Please upload 'Task1_Complete_Dataset.zip' generated from Task 1:")
uploaded = files.upload()
zip_name = next(iter(uploaded))

# Extract the zip file
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall("task1_data")

# Define paths
IMG_FOLDER = "task1_data/page_images"
JSON_FOLDER = "task1_data/vector_data"

# Verify pairing
images = sorted([f for f in os.listdir(IMG_FOLDER) if f.endswith('.png')])
jsons = sorted([f for f in os.listdir(JSON_FOLDER) if f.endswith('.json')])
print(f"✅ Success! Found {len(images)} images and {len(jsons)} JSON files.")

In [ ]:
# @title 3. Pre-Run Cost Estimation (Run this BEFORE the batch to see estimated cost)

def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split('([0-9]+)', s)]

# Scan all JSON files to understand the data profile
images_list = [f for f in os.listdir(IMG_FOLDER) if f.endswith('.png')]
images_list.sort(key=natural_sort_key)

print("📊 DATASET COST ANALYSIS")
print("=" * 60)

total_raw_tokens = 0
total_optimized_tokens = 0
tier_counts = {0: 0, 1: 0, 2: 0, 3: 0}
problem_sheets = []

for img_file in images_list:
    json_file = img_file.replace('.png', '.json')
    json_path = os.path.join(JSON_FOLDER, json_file)

    if os.path.exists(json_path):
        with open(json_path, 'r') as f:
            v_data = json.load(f)

        raw_size = len(json.dumps(v_data)) // 4
        # ACCOUNT FOR IMAGE: Every sheet includes both JSON text AND 1 image (258 tokens)
        total_raw_tokens += raw_size + 258

        # Count data components
        n_text = len(v_data.get('text', []))
        n_ocr = len(v_data.get('raster_ocr', []))
        n_geom = len(v_data.get('geometry', []))
        n_lines = len(v_data.get('detected_lines', []))

        # Estimate text-only size (what we actually send after pruning)
        text_only_data = {
            "text": v_data.get('text', []),
            "raster_ocr": v_data.get('raster_ocr', []),
        }
        text_tokens = len(json.dumps(text_only_data)) // 4

        # ACCOUNT FOR IMAGE: Add text size, safety cap + overhead, and the 258 image tile tokens
        total_optimized_tokens += min(text_tokens + 5000, 80000) + 258

        # Determine which tier this sheet would need
        if raw_size <= 80000:
            tier_counts[0] += 1
        elif n_lines > 5000:
            tier_counts[1] += 1
            if raw_size > 500000:
                problem_sheets.append((img_file, raw_size, n_lines, n_ocr))
        elif n_geom > 2000:
            tier_counts[2] += 1
        else:
            tier_counts[3] += 1

# Expected generation footprint baseline per analysis report
EST_OUTPUT_PER_SHEET = 2000

# --- REVISED TRUE PRE-RUN ESTIMATION MATH ---
# Points directly to the global variables we declared in Cell 1 ($0.50 input / $3.00 output)
raw_cost_input = (total_raw_tokens / 1_000_000) * TRUE_INPUT_PRICE_PER_1M
opt_cost_input = (total_optimized_tokens / 1_000_000) * TRUE_INPUT_PRICE_PER_1M
output_cost = (len(images_list) * EST_OUTPUT_PER_SHEET / 1_000_000) * TRUE_OUTPUT_PRICE_PER_1M

print(f"Total sheets: {len(images_list)}")
print(f"")
print(f"WITHOUT optimization:")
print(f"   Total input tokens:  ~{total_raw_tokens:,.0f}")
print(f"   Estimated input cost: ${raw_cost_input:.2f}")
print(f"   + output cost:        ${output_cost:.2f}")
print(f"   TOTAL estimated:      ${raw_cost_input + output_cost:.2f}")
print(f"")
print(f"WITH optimization:")
print(f"   Total input tokens:  ~{total_optimized_tokens:,.0f}")
print(f"   Estimated input cost: ${opt_cost_input:.2f}")
print(f"   + output cost:        ${output_cost:.2f}")
print(f"   TOTAL estimated:      ${opt_cost_input + output_cost:.2f}")
print(f"")
print(f"💰 Estimated savings: ${(raw_cost_input - opt_cost_input):.2f} ({((raw_cost_input - opt_cost_input)/max(raw_cost_input,0.01))*100:.0f}%)")
print(f"")
print(f"Pruning tier breakdown:")
print(f"   Tier 0 (no pruning needed): {tier_counts[0]} sheets")
print(f"   Tier 1 (lines removed):     {tier_counts[1]} sheets")
print(f"   Tier 2 (lines+geom):        {tier_counts[2]} sheets")
print(f"   Tier 3 (lines+geom+ocr):    {tier_counts[3]} sheets")

if problem_sheets:
    print(f"\n⚠️ Heaviest sheets (would have caused 400 errors without pruning):")
    for name, tokens, lines, ocr in sorted(problem_sheets, key=lambda x: -x[1])[:10]:
        print(f"   {name}: ~{tokens:,} tokens ({lines:,} lines, {ocr:,} OCR entries)")

In [ ]:
# @title 4. Helper Functions (Smart Pruning - Cost Optimized)

def estimate_tokens(text):
    """Rough token estimate: ~4 chars per token for English/JSON."""
    return len(text) // 4


def smart_prune(v_data, max_tokens=80000):
    """
    Cost-optimized multi-tier pruning.

    max_tokens=80000 keeps each API call cheap (~$0.01-0.02 input cost)
    while preserving 100% of text grounding data.

    Priority (highest to lowest value for LLM grounding):
      1. vector text     - exact text from PDF with positions (ALWAYS KEPT)
      2. raster_ocr      - OCR text from image (ALWAYS KEPT, filtered at Tier 3)
      3. geometry bbox   - bounding boxes (summarized at Tier 2+)
      4. detected_lines  - raw line coords (ALWAYS summarized - lowest value)

    Key insight: detected_lines are NEVER sent in full. Even for small sheets,
    we send a summary + samples. The LLM has the image for spatial reference.
    The text data is what prevents hallucination.
    """
    pruned = {
        "page": v_data.get("page"),
        "dims": v_data.get("dims"),
    }

    text_items = v_data.get("text", [])
    ocr_items = v_data.get("raster_ocr", [])
    geom_items = v_data.get("geometry", [])
    line_items = v_data.get("detected_lines", [])

    # --- ALWAYS: Replace detected_lines with a compact summary ---
    # This alone saves 50-90% of tokens on dense sheets
    pruned["detected_lines_summary"] = {
        "total_lines_detected": len(line_items),
        "note": "Line coordinates omitted. Refer to image for spatial/structural details.",
        "sample_lines": line_items[:5] if line_items else []
    }

    # --- Keep all vector text (this is the core grounding) ---
    pruned["text"] = text_items
    pruned["raster_ocr"] = ocr_items
    pruned["geometry"] = geom_items

    # Check if we fit in budget
    current_est = estimate_tokens(json.dumps(pruned))

    if current_est <= max_tokens:
        return pruned, 1  # Tier 1: lines summarized, everything else full

    # --- TIER 2: Summarize geometry ---
    large_geom = [g for g in geom_items
                  if abs(g["bbox"][2] - g["bbox"][0]) > 20
                  or abs(g["bbox"][3] - g["bbox"][1]) > 20]
    pruned["geometry"] = {
        "total_geometry_paths": len(geom_items),
        "large_elements_count": len(large_geom),
        "sample_large_elements": large_geom[:15],
        "note": "Geometry summarized. Refer to image for layout."
    }

    current_est = estimate_tokens(json.dumps(pruned))
    if current_est <= max_tokens:
        return pruned, 2

    # --- TIER 3: Filter OCR to high-confidence ---
    high_conf_ocr = [item for item in ocr_items if item.get("conf", 0) >= 0.3]
    # Sort by confidence descending and cap
    high_conf_ocr = sorted(high_conf_ocr, key=lambda x: x.get("conf", 0), reverse=True)
    if len(high_conf_ocr) > 400:
        high_conf_ocr = high_conf_ocr[:400]

    pruned["raster_ocr"] = high_conf_ocr
    pruned["ocr_note"] = f"Filtered to {len(high_conf_ocr)} of {len(ocr_items)} OCR entries (conf >= 0.3)."

    current_est = estimate_tokens(json.dumps(pruned))
    if current_est <= max_tokens:
        return pruned, 3

    # --- TIER 4 (Emergency): Also cap vector text if extremely dense ---
    # This should be very rare - only for sheets with massive embedded text
    if len(text_items) > 500:
        pruned["text"] = text_items[:500]
        pruned["text_note"] = f"Capped to 500 of {len(text_items)} text items."

    if len(high_conf_ocr) > 200:
        pruned["raster_ocr"] = high_conf_ocr[:200]

    return pruned, 4


print("✅ Helper functions loaded (cost-optimized, max 80K tokens per call).")

In [ ]:
# @title 5. Run Batch Analysis (Cost-Optimized + Checkpoint/Resume)

# --- THE UNIVERSAL MASTER PROMPT ---
MASTER_PROMPT = """
You are an expert GDOT (Georgia Department of Transportation) structural engineering analyst with deep knowledge of AASHTO standards, the GDOT Plan Development Process (PDP), and standard structural/civil drawing conventions.

You are analyzing a multi-layer "Digital Twin" document payload of a BRIDGE & CIVIL STRUCTURE Cover Sheet consisting of:
1. PNG Raster Image — Visual context and layout of the cover sheet.
2. Vector JSON — Direct, selectable text extracted with associated bounding box X/Y coordinates.
3. OCR JSON — Text recovered computationally from flattened or rasterized document layers (logos, seals, stamps).

Your task is to cross-reference ALL three data sources to produce a highly accurate, structured extraction of this bridge project cover sheet. Do not assume facts not explicitly supported by the data layers. If data is absent from all sources, explicitly flag it as [NOT FOUND]. For every single extracted property, you must provide a corresponding source token mapping exactly where the data was located ("vector" | "ocr" | "visual" | "NOT FOUND").

---

## 1. SHEET IDENTIFICATION
- Sheet number
- Drawing number / project code designation
- Document Type: Cover Sheet / Title Sheet
- Verified Project Engineering Domain: Bridge Project

## 2. PROJECT IDENTIFICATION
- Full project title and confirmed active Project P.I. Number (e.g., 331910 / 343455)
- County location and designated State Route (SR) / US Route / Road name
- Assigned GDOT District number
- Federal Aid Project Number (if explicitly present)
- Letting date or advertisement date
- Programmed fiscal year

## 3. RESPONSIBLE PARTIES
- Design firm, consulting engineering agency, or GDOT office of record
- Engineer of Record (EOR) full name and structural PE seal visibility status
- GDOT reviewer / approver name, title, and signature block status

## 4. QUANTITATIVE LOCATION DATA
- Specific State Route (SR) or US Route number
- Project Begin/End station boundaries or mileposts (e.g., STA XX+XX)
- Explicit GPS coordinates or county map reference vectors printed on the sheet

## 5. TITLE BLOCK VERIFICATION
- Active revision index table contents (dates, entities requesting updates, modified sheet logs)

## 6. FLAGS & ANOMALIES
- Missing or hidden PE stamp/signature
- Discrepancies where fields are visually clear in the image but missing from vector/OCR JSON data
- Skewed, rotated, or vertical text vectors, project logos, or state shields that raw computational OCR engines missed

## 7. EXPERT RISK & COORDINATION INSIGHTS
- Identify 3-5 specific construction risks if there are any visible or implied on this sheet.
- Target Examples: Excessive structural spans indicating unfeasible pier placements, utility/railroad right-of-way encroachments (e.g., CSX track interfaces), challenging hydraulic vertical clearance over historical stream floodplains, staging constraints for crane placement during girder installation, or discrepancies between structural station limits and civil roadway approach baselines.

---

## REQUIRED SUMMARY TABLE
- Create an Extraction Summary Table matching columns: Property Name | Extracted Value | Source Layer Used (Vector/OCR/Visual)
"""

# --- CONFIGURATION ---
RESULTS_FILE = "costOptimized_fullBatch_Greenville.md"
BASE_DELAY = 5
MAX_RETRIES = 5

# --- CHECKPOINT: Detect already-processed sheets ---
processed_sheets = set()
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, "r") as f:
        existing = f.read()
        processed_sheets = set(re.findall(r"# ANALYSIS: (.*?)\n", existing))
else:
    with open(RESULTS_FILE, "w") as f:
        f.write("# FULL PROJECT AUDIT REPORT\n\n")

# --- Get and sort images ---
images = [f for f in os.listdir(IMG_FOLDER) if f.endswith('.png')]
images.sort(key=natural_sort_key)
remaining = [img for img in images if img not in processed_sheets]

print(f"📊 Progress: {len(processed_sheets)} / {len(images)} sheets already complete.")
print(f"🚀 Processing {len(remaining)} remaining sheets...")


# --- Cost tracking (Updated to read true server responses) ---
total_input_tokens_sent = 0
total_output_tokens_received = 0  # Added to track generated context
total_sheets_processed = 0
consecutive_429s = 0
current_delay = BASE_DELAY
total_done = len(processed_sheets)

for idx, img_file in enumerate(remaining):
    json_file = img_file.replace('.png', '.json')
    img_path = os.path.join(IMG_FOLDER, img_file)
    json_path = os.path.join(JSON_FOLDER, json_file)

    if not os.path.exists(json_path):
        print(f"⚠️ No JSON found for {img_file}, skipping.")
        continue

    # Load data
    img_obj = Image.open(img_path)
    with open(json_path, 'r') as f:
        v_data = json.load(f)

    # Smart prune (cost-optimized)
    pruned_data, tier = smart_prune(v_data)
    json_payload = json.dumps(pruned_data)
    est_tokens = estimate_tokens(json_payload + MASTER_PROMPT)

    tier_label = ["Full", "Lines summarized", "Lines+Geom summarized",
                  "Lines+Geom+OCR filtered", "Emergency cap"][tier]
    print(f"\n[{total_done + idx + 1}/{len(images)}] {img_file}")
    print(f"  Tier {tier} ({tier_label}) | ~{est_tokens:,} tokens")

    # Retry loop
    success = False
    attempts = 0

    while not success and attempts < MAX_RETRIES:
        try:
            if attempts > 0:
                print(f"  Attempt {attempts + 1}...")

            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=[
                    f"{MASTER_PROMPT}\n\nGROUNDING DATA (JSON):\n{json_payload}",
                    img_obj
                ]
            )

           # Save immediately
            report_section = f"# ANALYSIS: {img_file}\n\n{response.text}\n\n---\n"
            with open(RESULTS_FILE, "a") as f:
                f.write(report_section)

            # =================================================================
            # 🛠️ EXTRACT METRICS STRAIGHT FROM GOOGLE'S SERVER RESPONSE
            # =================================================================
            success = True
            total_sheets_processed += 1
            consecutive_429s = 0
            current_delay = BASE_DELAY

            # Pull exact token transactions returned in the response usage headers
            if hasattr(response, 'usage_metadata') and response.usage_metadata:
                total_input_tokens_sent += response.usage_metadata.prompt_token_count
                total_output_tokens_received += response.usage_metadata.candidates_token_count
            else:
                # Emergency network fallback if metadata dropping occurs
                total_input_tokens_sent += est_tokens + 258  # accounts for prompt + image tile footprint
                total_output_tokens_received += 2000

            # Dynamic status update displayed every 10 sheets processed
            if total_sheets_processed % 10 == 0:
                run_in_cost = (total_input_tokens_sent / 1_000_000) * TRUE_INPUT_PRICE_PER_1M
                run_out_cost = (total_output_tokens_received / 1_000_000) * TRUE_OUTPUT_PRICE_PER_1M
                print(f"  💰 Running Ledger Total: ~${(run_in_cost + run_out_cost):.3f} for {total_sheets_processed} sheets")

            time.sleep(current_delay)

        except Exception as e:
            error_msg = str(e)

            if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg:
                consecutive_429s += 1
                wait_time = 60 + (consecutive_429s * 30)
                print(f"  ⚠️ Rate limit (429). Waiting {wait_time}s...")
                time.sleep(wait_time)
                current_delay = min(BASE_DELAY + (consecutive_429s * 3), 20)
                attempts += 1

            elif "400" in error_msg and "token" in error_msg.lower():
                print(f"  ❌ Token limit (400). Sending text-only fallback...")
                text_only = {
                    "page": v_data.get("page"),
                    "dims": v_data.get("dims"),
                    "text": v_data.get("text", [])[:300],
                    "raster_ocr": sorted(
                        [i for i in v_data.get("raster_ocr", []) if i.get("conf", 0) >= 0.5],
                        key=lambda x: x.get("conf", 0), reverse=True
                    )[:200],
                    "note": "Geometry omitted due to size. Rely on image for spatial analysis."
                }
                json_payload = json.dumps(text_only)
                est_tokens = estimate_tokens(json_payload + MASTER_PROMPT)
                print(f"  📉 Text-only: ~{est_tokens:,} tokens")
                attempts += 1

            else:
                print(f"  ❌ Error: {e}")
                with open(RESULTS_FILE, "a") as f:
                    f.write(f"# ANALYSIS: {img_file}\n\n"
                            f"**ERROR**: {error_msg}\n\n---\n")
                break

# =================================================================
# 📈 FINAL TRUE PRODUCTION ACCOUNTING AUDIT
# =================================================================
final_input_cost = (total_input_tokens_sent / 1_000_000) * TRUE_INPUT_PRICE_PER_1M
final_output_cost = (total_output_tokens_received / 1_000_000) * TRUE_OUTPUT_PRICE_PER_1M
true_total_calculated_cost = final_input_cost + final_output_cost

print(f"\n" + "=" * 60)
print(f"✅ BATCH OPERATION AUDIT COMPLETE (COST-OPTIMIZED)")
print(f"  Model Employed:                 {MODEL_NAME}")
print(f"  Sheets successfully written:    {total_sheets_processed}")
print(f"  True Input Context Tokens:      {total_input_tokens_sent:,}")
print(f"  True Output Generated Tokens:   {total_output_tokens_received:,}")
print(f"  ------------------------------------------------------------")
print(f"  Calculated Input Layer Cost:    ${final_input_cost:.4f}")
print(f"  Calculated Output Layer Cost:   ${final_output_cost:.4f}")
print(f"  🚨 TRUE CONSOLE LEDGER TOTAL:   ${true_total_calculated_cost:.3f}")
print(f"  Unoptimized Reference Cost:     ~$2.790") # Your baseline context benchmark
print(f"=" * 60)

In [ ]:
# @title 6. Download Report
from google.colab import files
files.download(RESULTS_FILE)

In [ ]:
# @title 7. View Report Preview (Optional)
from IPython.display import Markdown, display

if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, "r") as f:
        full_report = f.read()

    display(Markdown("## 📋 PROJECT AUDIT REPORT (Preview)\n"))
    if len(full_report) > 10000:
        display(Markdown(full_report[:10000000]))
        print(f"\n... (Showing first 10,000 chars of {len(full_report):,} total. Download the full report.)")
    else:
        display(Markdown(full_report))
else:
    print(f"⚠️ '{RESULTS_FILE}' not found. Run the analysis cell first.")